# Interactive Gillespie Module 1: Birth-Death Process

This notebook uses the smallest reaction network to build intuition for Gillespie SSA:

$$\varnothing \xrightarrow{k} X, \quad X \xrightarrow{\gamma X} \varnothing$$

The corresponding deterministic model is

$$\frac{dx}{dt}=k-\gamma x, \qquad x(t)=\frac{k}{\gamma}+\left(x_0-\frac{k}{\gamma}\right)e^{-\gamma t}.$$

Change the sliders to compare individual stochastic trajectories, their ensemble mean, the deterministic ODE curve, and the steady state $k/\gamma$. For this linear reaction network, the population mean is expected to follow the deterministic ODE.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams.update({
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [2]:
def gillespie_birth_death(k=5.0, gamma=0.15, x0=0, tmax=80.0, seed=None):
    """Exact SSA for: ∅ -> X with rate k; X -> ∅ with rate gamma*x."""
    rng = np.random.default_rng(seed)
    t = 0.0
    x = int(x0)
    T = [t]
    X = [x]

    while t < tmax:
        a1 = k
        a2 = gamma * x
        a0 = a1 + a2
        if a0 <= 0:
            break

        r1, r2 = rng.random(2)
        tau = -np.log(r1) / a0
        if t + tau > tmax:
            break

        t += tau
        if r2 * a0 < a1:
            x += 1
        else:
            x = max(0, x - 1)

        T.append(t)
        X.append(x)

    return np.array(T), np.array(X)

In [3]:
def deterministic_birth_death(t, k=5.0, gamma=0.15, x0=0):
    """Analytical solution of dx/dt = k - gamma*x."""
    t = np.asarray(t, dtype=float)
    return k / gamma + (x0 - k / gamma) * np.exp(-gamma * t)


def sample_step_trajectory(T, X, t_grid):
    """Sample a right-continuous SSA step trajectory on a common time grid."""
    indices = np.searchsorted(T, t_grid, side="right") - 1
    indices = np.clip(indices, 0, len(X) - 1)
    return X[indices]


def plot_birth_death(k=5.0, gamma=0.15, x0=0, tmax=80, n_traj=20, seed=1):
    fig, ax = plt.subplots(figsize=(8, 4))

    # A common grid is needed because different SSA trajectories have events
    # at different times.
    t_grid = np.linspace(0, tmax, 500)
    sampled_trajectories = []

    # Exact stochastic trajectories from Gillespie SSA
    for i in range(n_traj):
        T, X = gillespie_birth_death(k, gamma, x0, tmax, seed + i)
        sampled_trajectories.append(sample_step_trajectory(T, X, t_grid))
        ax.step(
            T, X, where="post", lw=1.0, alpha=max(0.12, min(0.55, 4 / n_traj)),
            label="Individual SSA cells" if i == 0 else None,
        )

    # Pointwise mean across all simulated cells
    mean_trajectory = np.mean(sampled_trajectories, axis=0)
    mean_color = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]
    ax.plot(
        t_grid, mean_trajectory, lw=3.0, color=mean_color,
        label=f"Mean of {n_traj} SSA cells",
    )

    # Deterministic ODE solution: dx/dt = k - gamma*x
    x_ode = deterministic_birth_death(t_grid, k, gamma, x0)
    ax.plot(
        t_grid, x_ode, color="black", ls="--", lw=2.4,
        label="Deterministic ODE",
    )

    # Long-time deterministic steady state
    ax.axhline(
        k / gamma, color="black", ls=":", lw=1.5, alpha=0.65,
        label=r"Steady state $k/\gamma$",
    )

    ax.set_xlim(0, tmax)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("Time")
    ax.set_ylabel("X molecule count")
    ax.set_title("The ensemble mean approaches the deterministic ODE")
    ax.legend(frameon=False, ncol=2)
    plt.show()


interact(
    plot_birth_death,
    k=FloatSlider(value=5.0, min=0.1, max=20, step=0.1, description="k"),
    gamma=FloatSlider(value=0.15, min=0.01, max=1.0, step=0.01, description="gamma"),
    x0=IntSlider(value=0, min=0, max=100, step=1, description="x0"),
    tmax=IntSlider(value=80, min=10, max=300, step=10, description="Tmax"),
    n_traj=IntSlider(value=20, min=1, max=200, step=1, description="n_cells"),
    seed=IntSlider(value=1, min=0, max=1000, step=1, description="seed"),
);


interactive(children=(FloatSlider(value=5.0, description='k', max=20.0, min=0.1), FloatSlider(value=0.15, desc…

In [4]:
def endpoint_distribution(k=5.0, gamma=0.15, x0=0, tmax=160, n_cells=1000, seed=1):
    rng = np.random.default_rng(seed)
    endpoints = []
    for i in range(n_cells):
        T, X = gillespie_birth_death(k, gamma, x0, tmax, int(rng.integers(0, 2**32-1)))
        endpoints.append(X[-1])

    endpoints = np.array(endpoints)
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.arange(endpoints.min() - 0.5, endpoints.max() + 1.5, 1)
    ax.hist(endpoints, bins=bins, density=True, alpha=0.8, edgecolor='white')
    ax.axvline(k/gamma, ls='--', lw=1.8, label='ODE steady state k/γ')
    ax.set_xlabel('X count at final time')
    ax.set_ylabel('probability density')
    ax.set_title('Many cells give a distribution')
    ax.legend(frameon=False)
    plt.show()

interact(
    endpoint_distribution,
    k=FloatSlider(value=5.0, min=0.1, max=20, step=0.1, description='k'),
    gamma=FloatSlider(value=0.15, min=0.01, max=1.0, step=0.01, description='gamma'),
    x0=IntSlider(value=0, min=0, max=100, step=1, description='x0'),
    tmax=IntSlider(value=160, min=20, max=500, step=20, description='Tmax'),
    n_cells=IntSlider(value=1000, min=50, max=5000, step=50, description='n_cells'),
    seed=IntSlider(value=1, min=0, max=1000, step=1, description='seed'),
);

interactive(children=(FloatSlider(value=5.0, description='k', max=20.0, min=0.1), FloatSlider(value=0.15, desc…